# Arabic Text Summarization Project Setup

In [36]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

import evaluate

from rouge_score import rouge_scorer


from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from rouge_score import rouge_scorer
rouge = evaluate.load("rouge")

import nltk
nltk.download("punkt", quiet=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset Loading & Exploration

In [37]:
dataset_path = "/kaggle/input/datasets/fadyelkbeer/arabic-text-summarization-30-000/wikiHow.csv"

df = pd.read_csv(dataset_path)

df = df[["text", "summary"]]

df.dropna(inplace=True)

df = df.sample(5000, random_state=42)

df.reset_index(drop=True, inplace=True)

df.shape

(5000, 2)

# Basic Statistics

In [38]:
df["doc_length"] = df["text"].apply(lambda x: len(str(x).split()))
df["summary_length"] = df["summary"].apply(lambda x: len(str(x).split()))

print(f"Average Document Length : {df['doc_length'].mean():.1f} words")
print(f"Average Summary Length  : {df['summary_length'].mean():.1f} words")

compression_ratio = (
    df["summary_length"] / df["doc_length"]
).mean()

print(f"Average Compression Ratio : {compression_ratio:.2%}")

Average Document Length : 346.3 words
Average Summary Length  : 30.1 words
Average Compression Ratio : inf%


# Data Preprocessing

## Split Dataset

In [39]:
train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42
)

In [40]:
train_df.shape, test_df.shape

((4500, 4), (500, 4))

## Model & Tokenizer

In [41]:
MODEL_NAME = "google/mt5-small"
print(f"Tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Tokenizer: google/mt5-small


## Custom Dataset

In [42]:
class ArabicSummarizationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_input_length=256,
        max_target_length=64
    ):

        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        source_text = "summarize Arabic: " + str(row["text"])
        target_text = str(row["summary"])

        source = self.tokenizer(
            source_text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        target = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target["input_ids"].squeeze()

        # Ignore padding tokens in loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": labels
        }

## Create Datasets

In [43]:
train_dataset = ArabicSummarizationDataset(
    train_df,
    tokenizer
)

test_dataset = ArabicSummarizationDataset(
    test_df,
    tokenizer
)

## Sample for check


In [44]:
sample = train_dataset[0]

print(f"Input IDs      : {sample['input_ids'].shape}")
print(f"Attention Mask : {sample['attention_mask'].shape}")
print(f"Labels         : {sample['labels'].shape}")

Input IDs      : torch.Size([256])
Attention Mask : torch.Size([256])
Labels         : torch.Size([64])


# Baseline Model Evaluation

## Load Pretrained Model

In [45]:
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print(f"Model Parameters: {baseline_model.num_parameters():,}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model Parameters: 556,291,456


## Function to generate summaries

In [48]:
def generate_summary(
    model,
    text,
    tokenizer,
    max_input_length=256,
    max_output_length=64
):

    text = "summarize Arabic: " + text

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_input_length,
            min_length=20,
            num_beams=8,
            no_repeat_ngram_size=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    summary = re.sub(r"<extra_id_\d+>", "", summary)

    summary = summary.strip()
    
    return summary

## Testing on 3 Samples

In [49]:
for i in range(3):

    document = str(test_df.iloc[i]["text"])
    reference_summary = str(test_df.iloc[i]["summary"])

    generated_summary = generate_summary(
        baseline_model,
        document,
        tokenizer
    )

    print(f"\n{'=' * 50}")
    print(f"Sample {i + 1}")
    print(f"{'=' * 50}")

    print("\n Document:")
    print(document[:200] + "...")

    print("\n Reference Summary:")
    print(reference_summary)

    print("\n Baseline Summary:")
    print(generated_summary)


Sample 1

 Document:
لا ينصح بوضعها في الثلاجة حيث لا تنضج ثمار الأفوكادو في درجة الحرارة المنخفضة. ربما تستغرق حتى 6 أيام كي تنضج.  يمكنك عمل هذه الطريقة إذا لم تستطيع أن تأكلها طازجة. بعد أن قضيت أياما تتابع فيها ثمرة ا...

 Reference Summary:
احفظ ثمرة الأفوكادو السليمة غير الناضجة في درجة حرارة الغرفة. * إذا لاحظت تحول الثمرة إلى اللون البني، فأنت لست في حاجة إلى التخلص منها، بل انزع الأجزاء ذات اللون البني واستخدم المتبقي منها قبل أن يتغير لونه. من الواضح أن أفضل حال لثمرة الأفوكادو عندما لا تكون مجمدة. راقب مراحل النضج.

 Baseline Summary:
على شرائح تفيد في أعداد أطباق السلطة والصلصة.

Sample 2

 Document:
يجب أن تداوم على هذا قبل كل مرة، خاصة وأنت في بداية رحلتك نحو الانتظام. يجب أن تعد عضلاتك جيدا، إذ لم تتعرض من قبل للضغط الناجم عن الركض المنتظم. جرب الإطالات من وضع الحركة في الإحماء.  عليك بالإطالات...

 Reference Summary:
قم بالإحماء الجيد لمدة من خمس إلى عشر دقائق قبل الركض. تنفس بعمق وبانتظام انتبه للطريقة التي تركض بها. اجعل خطواتك مناسبة لطبيعة الركض. ارخ الجزء العلوي م

# LORA FINE-TUNING

In [50]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## LoRA Configuration & Apply

In [51]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 344,064 || all params: 556,635,520 || trainable%: 0.0618


## Training Arguments

In [52]:
print(f"  Rank: {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Target Modules: {lora_config.target_modules}")
print(f"  Dropout: {lora_config.lora_dropout}")

  Rank: 8
  Alpha: 32
  Target Modules: {'q', 'v'}
  Dropout: 0.1


## Training Arguments

In [53]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    seed=42
)

## Data Collator

In [54]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

## Trainer

In [55]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

## Start Training

In [56]:
train_result = trainer.train()

print(f"\nFinal Training Loss: {train_result.training_loss:.4f}")

Epoch,Training Loss,Validation Loss
1,9.580492,7.508660
2,9.326426,7.337700
3,9.113976,7.287800



Final Training Loss: 11.3926


# Evaluation & Comparison

In [96]:
model.eval()

finetuned_summaries = []
references = []

for i in tqdm(range(20)):

    document = str(test_df.iloc[i]["text"])
    reference_summary = str(test_df.iloc[i]["summary"])

    generated_summary = generate_summary(
        model,
        document,
        tokenizer
    )

    finetuned_summaries.append(str(generated_summary).strip())
    references.append(str(reference_summary).strip())

100%|██████████| 20/20 [00:17<00:00,  1.17it/s]


## Generate Baseline Summaries

In [97]:
baseline_model.eval()

baseline_summaries = []

for i in tqdm(range(20)):

    document = str(test_df.iloc[i]["text"])

    generated_summary = generate_summary(
        baseline_model,
        document,
        tokenizer
    )

    baseline_summaries.append(
        str(generated_summary).strip()
    )

100%|██████████| 20/20 [00:08<00:00,  2.43it/s]


## ROUGE Evaluation

In [113]:
class ArabicTokenizer:
    def tokenize(self, text):
        return text.split()


def calculate_rouge(references, predictions):

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        tokenizer=ArabicTokenizer()
    )

    rouge1 = []
    rouge2 = []
    rougeL = []

    for ref, pred in zip(references, predictions):

        ref = str(ref).strip()
        pred = str(pred).strip()

        scores = scorer.score(ref, pred)

        rouge1.append(scores['rouge1'].fmeasure)
        rouge2.append(scores['rouge2'].fmeasure)
        rougeL.append(scores['rougeL'].fmeasure)

    return {
        "ROUGE-1": np.mean(rouge1),
        "ROUGE-2": np.mean(rouge2),
        "ROUGE-L": np.mean(rougeL)
    }

# Calculate Scores
baseline_scores = calculate_rouge(
    references,
    baseline_summaries
)

finetuned_scores = calculate_rouge(
    references,
    finetuned_summaries
)

## Comparison Table

In [114]:
comparison_df = pd.DataFrame({

    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],

    "Baseline": [
        baseline_scores["ROUGE-1"],
        baseline_scores["ROUGE-2"],
        baseline_scores["ROUGE-L"]
    ],

    "Fine-Tuned": [
        finetuned_scores["ROUGE-1"],
        finetuned_scores["ROUGE-2"],
        finetuned_scores["ROUGE-L"]
    ]
})

comparison_df["Improvement"] = (
    comparison_df["Fine-Tuned"] -
    comparison_df["Baseline"]
)

## Results

In [115]:
print("\n" + "=" * 60)
print("📈 BASELINE vs FINE-TUNED")
print("=" * 60)

print(comparison_df.round(6).to_string(index=False))


📈 BASELINE vs FINE-TUNED
 Metric  Baseline  Fine-Tuned  Improvement
ROUGE-1  0.056760    0.122322     0.065561
ROUGE-2  0.011510    0.032745     0.021236
ROUGE-L  0.045785    0.103499     0.057713


# Save Model & Tokenizer

In [66]:
output_dir = "./saved_model"

os.makedirs(output_dir, exist_ok=True)

print("Saving Fine-Tuned Model")
model.save_pretrained(output_dir)

# Save Tokenizer
tokenizer.save_pretrained(output_dir)

print(f"\nModel and Tokenizer Saved Successfully!")
print(f"Path: {output_dir}")

Saving Fine-Tuned Model

Model and Tokenizer Saved Successfully!
Path: ./saved_model


# Manual Inference

In [70]:
model.eval()

sample_text = test_df.iloc[5]["text"]

print("\n Original Document:\n")
print(sample_text[:1000])

print("\n" + "=" * 60)

generated_summary = generate_summary(
    model,
    sample_text,
    tokenizer
)

print("\n Generated Summary:\n")
print(generated_summary)


 Original Document:

هناك أطفال كأنهم كانوا يمسكون بكرة سلة في أيديهم أثناء ولادتهم، هؤلاء يكبرون ليصبحوا محترفي اللعبة. فمن الأفضل أن تبدأ صغيرا قدر الإمكان للحصول على الخبرة بقدر ما تستطيع. ابدأ صغيرا وسوف تتدفق لعبة كرة السلة في عروقك. البداية مع فريق المدرسة والحي شيء عظيم، ولكن راع حضور معسكرات تدريبية لكرة السلة مثل معسكر "فايف ستارز" والأكاديمية الوطنية لكرة السلة ومعسكر نخبة كرة السلة. مقابل بضع مئات من الدولارات كل موسم، ستستطيع التدرب مع الأفضل في منطقتك والبدء في تطوير مهارات ذات مستوى أعلى. حتي تتم ملاحظتك على مستوى الجامعات (الخطوة التالية)، تحتاج إلى اللمعان كالذهب في فريق مدرستك الثانوية. هذا لا يعني أن تصبح أنانيا -  في الواقع، "عدم" كونك لاعبا متعاونا مع زملائك يعتبر أمر ضدك. ذلك يعني ببساطة المخاطرة والتسديد والتمرير لزملائك وتحويل مجهوداتهم لنقاط. بالإضافة إلى أن تصبح لاعبا عظيما، يجب أيضا أن تكون متعاونا مع المدرب وسلسا في التعامل. إذا كنت لا تمنع اللاعبين الآخرين من اللعب بأفضل ما يمكنهم، فلن يساعدوك. وإذا كانت لديك نقطة ضعف يحاول المدرب تحسينها ولكن سلوكك لا يجعل